In [ ]:
# setup cv
import os
os.environ["MUJOCO_GL"] = "glfw"  # or "egl"

# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# robots
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun, log_rerun_data
from lerobot.record import record_loop

# env
from lerobot.envs.utils import env_to_policy_features

# torch
from torch import cuda

# utils
from dotenv import load_dotenv
from IPython.display import clear_output
import numpy as np
import time

# my code
from src.utils import scroll_print
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR
from envs.so101_env_utils import ACTIONS, DT, JOINTS, MUJOCO_DIR, JOINTS_MAX, JOINTS_MIN, SO101OBSTYPES, SO101TASKS

# my env
from envs.so101_env_config import SO101EnvConfig, make_so101_env
from src.robot_utils import sweep_leader_dataset, norm_modes_to_ranges

# set up os_env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Configuration:

In [2]:
PUSH_TO_HUB = False

### Connect Leader

In [3]:
teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

teleop = SO101Leader(teleop_config)

# get motor ranges
leader_ranges = norm_modes_to_ranges(teleop)
print('Joint space range for the actual robot:')
for name, (lo, hi) in zip(ACTIONS, leader_ranges):
    print(f"  {name:>12}: [{lo:.1f}, {hi:.1f}]")

Joint space range for the actual robot:
  shoulder_pan: [-100.0, 100.0]
  shoulder_lift: [-100.0, 100.0]
    elbow_flex: [-100.0, 100.0]
    wrist_flex: [-100.0, 100.0]
    wrist_roll: [-100.0, 100.0]
       gripper: [0.0, 100.0]


In [4]:
try:
    teleop.connect()
    print("Teleop device connected successfully.")
except Exception as e:
    print(e)


Could not connect on port 'COM8'. Make sure you are using the correct port.
Try running `python -m lerobot.find_port`



### Build Env

In [ ]:
env_cfg = SO101EnvConfig(
    task     = "TableLegAssembleTask",
    obs_type = "pixels_agent_pos",
    device = device,
    external_joint_ranges=leader_ranges
    )
env = make_so101_env(env_cfg, torch_actions = False)
scroll_print(env_cfg)

Left arrow key pressed. Exiting loop and rerecord the last episode...Left arrow key pressed. Exiting loop and rerecord the last episode...

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...Left arrow key pressed. Exiting loop and rerecord the last episode...

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key press

In [6]:
# # from lerobot.envs.utils import env_to_policy_features
# # env_to_policy_features(env_cfg)
# TODO handle datasets
# dataset.utils.build_dataset_frame

### Teleop Loop

In [39]:
synthetic_actions = sweep_leader_dataset(leader_ranges, 100, 20)
synthetic_actions = np.zeros_like(synthetic_actions)

In [ ]:
# init rr + kb listener
listener, events = init_keyboard_listener()
_init_rerun(session_name="teleop_env1", recording_id=time.time())

done = False
frame_time = 1.0 / env_cfg.fps
step_idx = 0

while not done and step_idx < len(synthetic_actions):
    loop_start = time.time()
    
    # Check keyboard flags
    if events["exit_early"]:
        pass # TODO implement
    if events["rerecord_episode"]:
        pass # TODO implement
    if events["stop_recording"]:
        print("Stopping data recording.")
        break
    
    # 1. Get leader action, map to env space
    # leader_action = teleop.get_action() TODO temporary for testing
    leader_action = synthetic_actions[step_idx]
    step_idx += 1
    
    # 2. Step env
    obs, reward, done, truncated, info = env.step(leader_action)
    
    # 3. Log to rerun
    obs_to_log = {
        k: (v.detach().cpu().squeeze(0).permute(1, 2, 0).numpy().clip(0, 255) * 255).astype(np.uint8)
        if "images" in k else v.detach().cpu().squeeze(0).float().numpy()
        for k, v in obs.items()
    }
    obs_to_log["render"]= env.render()
    action_to_log = {"action": np.array(leader_action)}
    log_rerun_data(obs_to_log, action_to_log)
    
    # 4. log to dataset
    pass
    
    # 5. Maintain FPS
    elapsed = time.time() - loop_start
    time.sleep(max(0, frame_time - elapsed))
    elapsed = time.time() - loop_start
    actual_fps = 1.0 / elapsed if elapsed > 0 else float("inf")
    clear_output(wait=True)
    print(f"Target FPS: {env_cfg.fps:.1f}, Actual FPS: {actual_fps:.2f}")

# stop listener when done
if listener:
    listener.stop()

In [ ]:
teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()